### Libraries

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.preprocessing import StandardScaler

In [4]:
class HeirarchicalLSTM(Dataset):
    def __init__(self, original_data, hourly_data, daily_data, weekly_data):
        self.original_data = torch.FloatTensor(original_data)
        self.hourly_data = torch.FloatTensor(hourly_data)
        self.daily_data = torch.FloatTensor(daily_data)
        self.weekly_data = torch.FloatTensor(weekly_data)

    def __len__(self):
        return len(self.original_data)
    
    def __getitem__(self, idx):
        return {
            'wsb': self.original_data[idx],
            'hourly': self.hourly_data[idx],
            'daily': self.daily_data[idx], 
            'weekly': self.weekly_data[idx],
        }

In [ ]:
class HourlyLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, self.hidden_size, self.num_layers, batch_first=True)

        # Attention mechanism to focus on most suspicious hours
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )
        # Output projection to create hourly embedding
        self.output_projection = nn.Linear(hidden_size, hidden_size // 2)
        self.dropout = nn.Dropout(dropout)